In [1]:
import numpy as np
import networkx as nx
import cvxpy as cp


# =========================
# 1. Generate graph
# =========================
def generate_graph(n=50, p=0.3, seed=0):
    return nx.erdos_renyi_graph(n, p, seed=seed)


# =========================
# 2. Solve SDP relaxation
# =========================
def solve_sdp(G):
    n = G.number_of_nodes()
    X = cp.Variable((n, n), symmetric=True)

    constraints = [X >> 0, cp.diag(X) == 1]

    edges = np.array(list(G.edges()))
    i_idx = edges[:, 0]
    j_idx = edges[:, 1]

    obj = cp.sum((1 - X[i_idx, j_idx]) / 2)

    prob = cp.Problem(cp.Maximize(obj), constraints)
    prob.solve(solver=cp.SCS)

    return X.value, prob.value


# =========================
# 3. Factor SDP solution
# =========================
def factorize_sdp(X):
    eigvals, eigvecs = np.linalg.eigh(X)
    eigvals = np.clip(eigvals, 0, None)
    return eigvecs @ np.diag(np.sqrt(eigvals))


# =========================
# 4. Hyperplane rounding
# =========================
def gw_rounding(G, V, num_samples=100):
    n, d = V.shape

    best_cut = 0
    best_partition = None

    edges = np.array(list(G.edges()))

    for _ in range(num_samples):
        r = np.random.randn(d)
        r /= np.linalg.norm(r)

        signs = np.sign(V @ r)

        cut = np.sum(signs[edges[:, 0]] != signs[edges[:, 1]])

        if cut > best_cut:
            best_cut = cut
            best_partition = signs

    return best_cut, best_partition


# =========================
# 5. Full pipeline
# =========================
def goemans_williamson(G):

    print("Solving SDP...")
    X, sdp_val = solve_sdp(G)

    print("Rounding...")
    V = factorize_sdp(X)

    cut, partition = gw_rounding(G, V)

    print("\nResults:")
    print(f"SDP value: {sdp_val:.2f}")
    print(f"Cut value: {cut}")
    print(f"Approx ratio: {cut / sdp_val:.3f}")

    return cut, partition


# =========================
# RUN
# =========================
if __name__ == "__main__":
    G = generate_graph(n=50, p=0.3)
    goemans_williamson(G)

Solving SDP...
Rounding...

Results:
SDP value: 250.24
Cut value: 243
Approx ratio: 0.971
